#ChatBot 1 - Mobilidade Urbana
##Transformers + Embeddings

In [1]:
import random
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

#base de treinamento

In [2]:
dados = [
    # Transporte público
    ("Como melhorar o transporte público?", "transporte_publico"),
    ("O ônibus demora muito na minha cidade", "transporte_publico"),
    ("Quais são os problemas do transporte coletivo?", "transporte_publico"),
    ("O que é transporte público de qualidade?", "transporte_publico"),
    ("Como tornar os ônibus mais eficientes?", "transporte_publico"),
    ("Por que investir em transporte coletivo?", "transporte_publico"),

    # Ciclovias
    ("Qual a importância das ciclovias?", "ciclovias"),
    ("Andar de bicicleta ajuda na mobilidade urbana?", "ciclovias"),
    ("Como incentivar o uso da bicicleta?", "ciclovias"),
    ("Ciclovias reduzem o trânsito?", "ciclovias"),
    ("A cidade precisa de mais ciclofaixas", "ciclovias"),
    ("Bicicletas são boas para o deslocamento urbano?", "ciclovias"),

    # Congestionamento
    ("Como reduzir os congestionamentos?", "congestionamento"),
    ("O trânsito está cada vez pior", "congestionamento"),
    ("Por que há tantos engarrafamentos?", "congestionamento"),
    ("Como diminuir o excesso de carros?", "congestionamento"),
    ("O que causa congestionamento nas cidades?", "congestionamento"),
    ("Quais soluções existem para o trânsito pesado?", "congestionamento"),

    # Sustentabilidade
    ("Mobilidade urbana pode ser sustentável?", "sustentabilidade"),
    ("Como reduzir a poluição no transporte?", "sustentabilidade"),
    ("Transporte sustentável é importante?", "sustentabilidade"),
    ("Como a mobilidade afeta o meio ambiente?", "sustentabilidade"),
    ("Carros poluem muito as cidades", "sustentabilidade"),
    ("Quais transportes são mais sustentáveis?", "sustentabilidade"),

    # Acessibilidade
    ("Como melhorar a acessibilidade no transporte?", "acessibilidade"),
    ("Ônibus precisam ser acessíveis para cadeirantes", "acessibilidade"),
    ("Pessoas com deficiência têm dificuldade de locomoção", "acessibilidade"),
    ("O que é mobilidade urbana inclusiva?", "acessibilidade"),
    ("Como garantir acessibilidade nas calçadas?", "acessibilidade"),
    ("A cidade deve ser acessível para todos", "acessibilidade"),

    # Segurança
    ("Como melhorar a segurança no trânsito?", "seguranca"),
    ("Muitos acidentes acontecem nas ruas", "seguranca"),
    ("Pedestres precisam de mais segurança", "seguranca"),
    ("Como proteger ciclistas no trânsito?", "seguranca"),
    ("O que fazer para reduzir acidentes?", "seguranca"),
    ("Faixas de pedestres ajudam na segurança?", "seguranca"),

    # Calçadas
    ("As calçadas estão ruins", "calcadas"),
    ("Como melhorar as calçadas da cidade?", "calcadas"),
    ("Calçadas acessíveis são importantes?", "calcadas"),
    ("Buracos nas calçadas atrapalham os pedestres", "calcadas"),
    ("A cidade precisa cuidar melhor das calçadas", "calcadas"),
    ("Por que as calçadas são importantes na mobilidade?", "calcadas"),

    # Integração
    ("O que é integração entre transportes?", "integracao"),
    ("Como integrar ônibus, metrô e bicicleta?", "integracao"),
    ("A integração facilita a mobilidade urbana?", "integracao"),
    ("Como melhorar a conexão entre transportes?", "integracao"),
    ("Bilhete único ajuda na mobilidade?", "integracao"),
    ("Transporte integrado melhora a cidade?", "integracao"),

    # Tarifas
    ("A passagem de ônibus está cara", "tarifas"),
    ("Como reduzir o preço do transporte público?", "tarifas"),
    ("Tarifa zero é uma boa ideia?", "tarifas"),
    ("O valor da passagem afeta os usuários?", "tarifas"),
    ("Por que o transporte público custa caro?", "tarifas"),
    ("Subsídio no transporte público é importante?", "tarifas"),

    # Planejamento urbano
    ("Como o planejamento urbano afeta a mobilidade?", "planejamento_urbano"),
    ("A cidade precisa ser melhor planejada", "planejamento_urbano"),
    ("Bairros distantes dificultam o deslocamento", "planejamento_urbano"),
    ("O crescimento urbano causa problemas no trânsito?", "planejamento_urbano"),
    ("Como organizar melhor a cidade?", "planejamento_urbano"),
    ("Planejamento urbano ajuda na mobilidade?", "planejamento_urbano"),
]

df = pd.DataFrame(dados, columns=["pergunta", "intencao"])
df.head()

,pergunta,intencao
0,Como melhorar o transporte público?,transporte_publico
1,O ônibus demora muito na minha cidade,transporte_publico
2,Quais são os problemas do transporte coletivo?,transporte_publico
3,O que é transporte público de qualidade?,transporte_publico
4,Como tornar os ônibus mais eficientes?,transporte_publico


#respostas do chatbot

In [3]:
respostas = {
    "transporte_publico": [
        "O transporte público é essencial para melhorar a mobilidade urbana, pois reduz a dependência de carros particulares e amplia o acesso da população à cidade.",
        "Para melhorar o transporte público, é importante investir em frequência, conforto, acessibilidade, segurança e integração entre linhas.",
        "Um transporte coletivo eficiente contribui para reduzir congestionamentos, poluição e desigualdades no acesso aos espaços urbanos."
    ],

    "ciclovias": [
        "As ciclovias incentivam o uso da bicicleta como meio de transporte, tornando os deslocamentos mais sustentáveis e saudáveis.",
        "Investir em ciclovias ajuda a reduzir o número de carros nas ruas e melhora a segurança dos ciclistas.",
        "A bicicleta pode ser uma alternativa importante para trajetos curtos, especialmente quando há infraestrutura adequada."
    ],

    "congestionamento": [
        "Os congestionamentos geralmente são causados pelo excesso de veículos particulares, falta de planejamento urbano e baixa qualidade do transporte público.",
        "Para reduzir congestionamentos, é necessário investir em transporte coletivo, ciclovias, calçadas adequadas e integração entre modais.",
        "Diminuir a dependência do carro é uma das principais estratégias para melhorar o trânsito nas cidades."
    ],

    "sustentabilidade": [
        "A mobilidade urbana sustentável busca reduzir impactos ambientais, priorizando transporte coletivo, bicicletas, caminhadas e veículos menos poluentes.",
        "Transportes sustentáveis ajudam a diminuir a emissão de poluentes e melhoram a qualidade de vida nas cidades.",
        "Uma cidade sustentável precisa oferecer alternativas reais ao uso excessivo do carro particular."
    ],

    "acessibilidade": [
        "A acessibilidade é fundamental para garantir que todas as pessoas possam circular pela cidade com autonomia e segurança.",
        "Uma mobilidade urbana inclusiva precisa considerar pessoas com deficiência, idosos, crianças e pessoas com mobilidade reduzida.",
        "Calçadas adequadas, rampas, sinalização e transporte adaptado são elementos importantes para a acessibilidade urbana."
    ],

    "seguranca": [
        "A segurança no trânsito depende de boa sinalização, fiscalização, educação no trânsito e infraestrutura adequada para pedestres e ciclistas.",
        "Reduzir acidentes exige políticas públicas que priorizem a vida, especialmente de pedestres, ciclistas e usuários do transporte coletivo.",
        "Faixas de pedestres, ciclovias protegidas e redução da velocidade em áreas urbanas ajudam a tornar o trânsito mais seguro."
    ],

    "calcadas": [
        "As calçadas são parte essencial da mobilidade urbana, pois todo deslocamento começa ou termina com o pedestre.",
        "Calçadas bem cuidadas, largas e acessíveis melhoram a circulação das pessoas e tornam a cidade mais inclusiva.",
        "Buracos, obstáculos e falta de rampas dificultam a locomoção e prejudicam principalmente idosos e pessoas com deficiência."
    ],

    "integracao": [
        "A integração entre transportes permite que o usuário combine ônibus, metrô, bicicleta e caminhada de forma mais eficiente.",
        "Sistemas integrados reduzem o tempo de deslocamento e tornam o transporte público mais atrativo.",
        "Bilhete único, terminais integrados e ciclovias conectadas ao transporte coletivo são exemplos de integração na mobilidade urbana."
    ],

    "tarifas": [
        "O preço da passagem influencia diretamente o acesso da população ao transporte público.",
        "Tarifas muito altas podem excluir pessoas de oportunidades de trabalho, estudo, saúde e lazer.",
        "Políticas como subsídios, integração tarifária e tarifa social podem tornar o transporte mais acessível."
    ],

    "planejamento_urbano": [
        "O planejamento urbano influencia a mobilidade porque define onde as pessoas moram, trabalham, estudam e acessam serviços.",
        "Cidades muito espalhadas aumentam a necessidade de longos deslocamentos e dificultam o transporte público eficiente.",
        "Um bom planejamento urbano aproxima moradia, trabalho, lazer e serviços, reduzindo a dependência do carro."
    ]
}

#Separar treino e teste

In [4]:
X = df["pergunta"]
y = df["intencao"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

#criação do modelo com TF-IDF + Naive Bayes

In [5]:
modelo = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2)
    )),
    ("naive_bayes", MultinomialNB())
])

modelo.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(ngram_range=(1, 2), strip_accents='unicode')),
                ('naive_bayes', MultinomialNB())])

#avalair o modelo

In [6]:
y_pred = modelo.predict(X_test)

print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório de classificação:\n")
print(classification_report(y_test, y_pred))

Acurácia: 0.13333333333333333

Relatório de classificação:

                     precision    recall  f1-score   support

     acessibilidade       0.00      0.00      0.00         1
           calcadas       0.50      1.00      0.67         1
          ciclovias       0.00      0.00      0.00         2
   congestionamento       0.00      0.00      0.00         2
         integracao       0.00      0.00      0.00         2
planejamento_urbano       0.00      0.00      0.00         1
          seguranca       0.00      0.00      0.00         2
   sustentabilidade       0.00      0.00      0.00         1
            tarifas       0.20      1.00      0.33         1
 transporte_publico       0.00      0.00      0.00         2

           accuracy                           0.13        15
          macro avg       0.07      0.20      0.10        15
       weighted avg       0.05      0.13      0.07        15



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#function

In [7]:
def chatbot(pergunta_usuario):
    intencao_prevista = modelo.predict([pergunta_usuario])[0]

    probabilidades = modelo.predict_proba([pergunta_usuario])[0]
    confianca = max(probabilidades)

    if confianca < 0.30:
        return {
            "pergunta": pergunta_usuario,
            "intencao": "desconhecida",
            "resposta": "Desculpe, não entendi bem sua pergunta. Você pode reformular falando sobre transporte público, ciclovias, trânsito, acessibilidade ou sustentabilidade?"
        }

    resposta = random.choice(respostas[intencao_prevista])

    return {
        "pergunta": pergunta_usuario,
        "intencao": intencao_prevista,
        "resposta": resposta
    }

#execucao/testes

In [8]:
testes = [
    "Como melhorar o trânsito da cidade?",
    "A passagem de ônibus está muito cara",
    "Por que precisamos de ciclovias?"
]

for pergunta in testes:
    resultado = chatbot(pergunta)
    print("Pergunta:", resultado["pergunta"])
    print("Intenção identificada:", resultado["intencao"])
    print("Resposta:", resultado["resposta"])
    print("-" * 50)

Pergunta: Como melhorar o trânsito da cidade?
Intenção identificada: desconhecida
Resposta: Desculpe, não entendi bem sua pergunta. Você pode reformular falando sobre transporte público, ciclovias, trânsito, acessibilidade ou sustentabilidade?
--------------------------------------------------
Pergunta: A passagem de ônibus está muito cara
Intenção identificada: desconhecida
Resposta: Desculpe, não entendi bem sua pergunta. Você pode reformular falando sobre transporte público, ciclovias, trânsito, acessibilidade ou sustentabilidade?
--------------------------------------------------
Pergunta: Por que precisamos de ciclovias?
Intenção identificada: desconhecida
Resposta: Desculpe, não entendi bem sua pergunta. Você pode reformular falando sobre transporte público, ciclovias, trânsito, acessibilidade ou sustentabilidade?
--------------------------------------------------


In [ ]:
#conversa interativa

In [9]:
print("Chatbot sobre Mobilidade Urbana")
print("Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Você: ")

    if pergunta.lower() in ["sair", "exit", "encerrar", "parar"]:
        print("Chatbot: Até mais! Continue refletindo sobre uma mobilidade urbana mais justa e sustentável.")
        break

    resultado = chatbot(pergunta)

    print("Intenção identificada:", resultado["intencao"])
    print("Chatbot:", resultado["resposta"])

Chatbot sobre Mobilidade Urbana
Digite 'sair' para encerrar.

Você: Para que ter ciclovia?
Intenção identificada: desconhecida
Chatbot: Desculpe, não entendi bem sua pergunta. Você pode reformular falando sobre transporte público, ciclovias, trânsito, acessibilidade ou sustentabilidade?
Você: Sair
Chatbot: Até mais! Continue refletindo sobre uma mobilidade urbana mais justa e sustentável.


#ChatBot 2 - Sistema de orientação emergencial
##Transformers + Embeddings

In [24]:
!pip -q install sentence-transformers

In [25]:
import random
import pandas as pd
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# criação das intenções e frases do treinamento

In [26]:
dados_emergencia = [
    ("Minha rua está alagada, o que devo fazer?", "alagamento"),
    ("A água está entrando na minha casa", "alagamento"),
    ("Como agir em caso de enchente?", "alagamento"),
    ("Estou preso em uma área alagada", "alagamento"),

    ("Há risco de deslizamento perto da minha casa", "deslizamento"),
    ("A encosta está rachando depois da chuva", "deslizamento"),
    ("O barranco perto de casa pode cair", "deslizamento"),
    ("O que fazer se houver deslizamento de terra?", "deslizamento"),

    ("Tem um incêndio no prédio", "incendio"),
    ("Senti cheiro de fumaça dentro de casa", "incendio"),
    ("Como agir em caso de incêndio?", "incendio"),
    ("Há fogo em uma residência próxima", "incendio"),

    ("Está tendo uma tempestade muito forte", "tempestade"),
    ("Como me proteger durante raios e ventos fortes?", "tempestade"),
    ("O que fazer durante uma chuva intensa?", "tempestade"),
    ("Está ventando muito e tenho medo de sair", "tempestade"),

    ("Uma pessoa passou mal perto de mim", "primeiros_socorros"),
    ("Alguém caiu e está ferido", "primeiros_socorros"),
    ("O que fazer se alguém desmaiar?", "primeiros_socorros"),
    ("Tem uma vítima precisando de ajuda médica", "primeiros_socorros"),

    ("Preciso sair de casa por causa do risco", "evacuacao"),
    ("Para onde ir em caso de emergência?", "evacuacao"),
    ("Como agir se mandarem evacuar a área?", "evacuacao"),
    ("Onde procurar abrigo em uma situação de risco?", "evacuacao"),
]

df_emergencia = pd.DataFrame(dados_emergencia, columns=["frase", "intencao"])

#respostas

In [27]:
respostas_emergencia = {
    "alagamento": "Em caso de alagamento, evite atravessar áreas com correnteza, procure um local seguro e acione a Defesa Civil pelo 199 ou os Bombeiros pelo 193 se houver risco imediato.",

    "deslizamento": "Em caso de risco de deslizamento, saia da área afetada, evite ficar próximo a encostas, muros ou rachaduras e acione a Defesa Civil pelo 199.",

    "incendio": "Em caso de incêndio, saia do local com segurança, não use elevadores, evite inalar fumaça e acione imediatamente o Corpo de Bombeiros pelo 193.",

    "tempestade": "Durante tempestades fortes, permaneça em local seguro, evite árvores, postes, áreas abertas e locais sujeitos a alagamento.",

    "primeiros_socorros": "Se alguém estiver ferido ou passando mal, mantenha a calma, não movimente a vítima sem necessidade e acione o SAMU pelo 192 ou os Bombeiros pelo 193.",

    "evacuacao": "Se houver orientação de evacuação, saia com calma, leve apenas itens essenciais e siga as rotas indicadas pelas autoridades ou pela Defesa Civil."
}

#carregament odo transform

In [28]:
# classificação zero-shot
modelo_transformer = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

modelo_embedding = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#geraando embeddings das frases do treinamento

In [30]:
lista_intencoes = list(respostas_emergencia.keys())

frases_treinamento = df_emergencia["frase"].tolist()
intencoes_treinamento = df_emergencia["intencao"].tolist()

embeddings_treinamento = modelo_embedding.encode(
    frases_treinamento,
    convert_to_tensor=True
)

#função do chatbot

In [33]:
def chatbot_emergencia(pergunta_usuario):
    # 1ª tentativa: classificação zero-shot com Transformer
    resultado_transformer = modelo_transformer(
        pergunta_usuario,
        candidate_labels=lista_intencoes
    )

    intencao_transformer = resultado_transformer["labels"][0]
    confianca_transformer = resultado_transformer["scores"][0]

    # Se a confiança do Transformer for boa, usa essa intenção
    if confianca_transformer >= 0.50:
        return {
            "pergunta": pergunta_usuario,
            "modelo_usado": "Transformer zero-shot",
            "intencao": intencao_transformer,
            "confianca": round(confianca_transformer, 2),
            "resposta": respostas_emergencia[intencao_transformer]
        }

    # 2ª tentativa: fallback com embeddings e similaridade de cosseno
    embedding_usuario = modelo_embedding.encode(
        pergunta_usuario,
        convert_to_tensor=True
    )

    similaridades = util.cos_sim(
        embedding_usuario,
        embeddings_treinamento
    )[0]

    indice_mais_proximo = similaridades.argmax().item()
    confianca_embedding = similaridades[indice_mais_proximo].item()
    intencao_embedding = intencoes_treinamento[indice_mais_proximo]

    if confianca_embedding >= 0.35:
        return {
            "pergunta": pergunta_usuario,
            "modelo_usado": "Embeddings semânticos",
            "intencao": intencao_embedding,
            "confianca": round(confianca_embedding, 2),
            "resposta": respostas_emergencia[intencao_embedding]
        }

    return {
        "pergunta": pergunta_usuario,
        "modelo_usado": "Fallback padrão",
        "intencao": "desconhecida",
        "confianca": round(max(confianca_transformer, confianca_embedding), 2),
        "resposta": "Desculpe, não consegui identificar a situação. Em caso de emergência real, acione os serviços oficiais: Defesa Civil 199, Bombeiros 193, SAMU 192 ou Polícia Militar 190."
    }

#testes

In [34]:
testes_emergencia = [
    "Minha casa está enchendo de água por causa da chuva",
    "Tem fumaça saindo de um apartamento",
    "Uma pessoa caiu e parece estar desacordada"
]

for pergunta in testes_emergencia:
    resultado = chatbot_emergencia(pergunta)

    print("Pergunta:", resultado["pergunta"])
    print("Modelo usado:", resultado["modelo_usado"])
    print("Intenção identificada:", resultado["intencao"])
    print("Confiança:", resultado["confianca"])
    print("Resposta:", resultado["resposta"])
    print("-" * 70)

Pergunta: Minha casa está enchendo de água por causa da chuva
Modelo usado: Embeddings semânticos
Intenção identificada: alagamento
Confiança: 0.82
Resposta: Em caso de alagamento, evite atravessar áreas com correnteza, procure um local seguro e acione a Defesa Civil pelo 199 ou os Bombeiros pelo 193 se houver risco imediato.
----------------------------------------------------------------------
Pergunta: Tem fumaça saindo de um apartamento
Modelo usado: Embeddings semânticos
Intenção identificada: incendio
Confiança: 0.59
Resposta: Em caso de incêndio, saia do local com segurança, não use elevadores, evite inalar fumaça e acione imediatamente o Corpo de Bombeiros pelo 193.
----------------------------------------------------------------------
Pergunta: Uma pessoa caiu e parece estar desacordada
Modelo usado: Transformer zero-shot
Intenção identificada: deslizamento
Confiança: 0.81
Resposta: Em caso de risco de deslizamento, saia da área afetada, evite ficar próximo a encostas, muros o

#conversa

In [35]:
print("Chatbot de Orientação Emergencial")
print("Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Você: ")

    if pergunta.lower() in ["sair", "exit", "encerrar", "parar"]:
        print("Chatbot: Encerrando atendimento. Em caso de emergência real, acione os serviços oficiais.")
        break

    resultado = chatbot_emergencia(pergunta)

    print("Modelo usado:", resultado["modelo_usado"])
    print("Intenção identificada:", resultado["intencao"])
    print("Confiança:", resultado["confianca"])
    print("Chatbot:", resultado["resposta"])
    print()

Chatbot de Orientação Emergencial
Digite 'sair' para encerrar.

Você: ta ficando quente e com fumaça aqui
Modelo usado: Embeddings semânticos
Intenção identificada: incendio
Confiança: 0.58
Chatbot: Em caso de incêndio, saia do local com segurança, não use elevadores, evite inalar fumaça e acione imediatamente o Corpo de Bombeiros pelo 193.

Você: ta ficando quente e dificil de respirar aqui
Modelo usado: Embeddings semânticos
Intenção identificada: deslizamento
Confiança: 0.59
Chatbot: Em caso de risco de deslizamento, saia da área afetada, evite ficar próximo a encostas, muros ou rachaduras e acione a Defesa Civil pelo 199.

Você: a pessoa ta deitada desmaiada e vomitou
Modelo usado: Transformer zero-shot
Intenção identificada: deslizamento
Confiança: 0.8
Chatbot: Em caso de risco de deslizamento, saia da área afetada, evite ficar próximo a encostas, muros ou rachaduras e acione a Defesa Civil pelo 199.

Você: a pessoa ta sangrando muito e ficando branca
Modelo usado: Embeddings semâ

##Fechamento

O chatbot 1, usando com TF-IDF e Naive Bayes, apresentou bons resultados apenas quando as perguntas dos usuários eram parecidas com as frases usadas no treinamento.

O chatbot 2, usando com Transformers e embeddings, apresentou melhor compreensão de perguntas escritas de formas diferentes, pois compara o significado das frases, e não apenas a presença de palavras semelhantes. Por isso, demonstrou maior capacidade de reconhecer intenções.